# ⛰️ Day 06: Heaps & Top-K Problems

---

## When Sorting Is Overkill

Someone asks "find the 3rd largest element." You could sort the whole array — O(n log n). But why sort EVERYTHING when you only care about 3 elements?

That's where heaps come in. A **min-heap** of size K gives you the K largest elements in O(n log k) — much better when k << n.

**Python's `heapq` is a min-heap by default.** To simulate a max-heap, negate the values.

```
Quick cheat sheet:
- heapq.heappush(heap, val)      → O(log n)
- heapq.heappop(heap)            → O(log n), returns smallest
- heapq.heapify(list)            → O(n), converts list to heap
- heapq.nlargest(k, iterable)    → O(n log k)
- heapq.nsmallest(k, iterable)   → O(n log k)
```

---

In [ ]:
import heapq

# Min-heap basics
heap = []
heapq.heappush(heap, 5)
heapq.heappush(heap, 2)
heapq.heappush(heap, 8)
heapq.heappush(heap, 1)

print(f"Heap: {heap}")           # [1, 2, 8, 5] — internal order, but pop gives smallest
print(f"Pop: {heapq.heappop(heap)}")  # 1
print(f"Pop: {heapq.heappop(heap)}")  # 2

# Max-heap trick: negate values
max_heap = []
for val in [5, 2, 8, 1]:
    heapq.heappush(max_heap, -val)

print(f"Max pop: {-heapq.heappop(max_heap)}")  # 8

---

## 🎯 Problem 1: Kth Largest Element in an Array (LC #215)

THE heap interview question. Everyone gets asked this.

```
Input: [3, 2, 1, 5, 6, 4], k = 2  →  Output: 5
```

**Two approaches:**
1. Min-heap of size k — O(n log k)
2. Quickselect — O(n) average (but O(n²) worst case)

In [ ]:
import heapq

def findKthLargest(nums, k):
    """
    Min-heap of size k: the root is the kth largest.
    Time: O(n log k), Space: O(k)
    """
    heap = []  # min-heap
    
    for num in nums:
        heapq.heappush(heap, num)
        if len(heap) > k:       # Keep only k elements
            heapq.heappop(heap)  # Remove smallest → leaves k largest
    
    return heap[0]  # Root = smallest of the k largest = kth largest

print(findKthLargest([3, 2, 1, 5, 6, 4], 2))  # 5
print(findKthLargest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4))  # 4

# One-liner version:
print(heapq.nlargest(2, [3, 2, 1, 5, 6, 4])[-1])  # 5

---

## 🎯 Problem 2: Top K Frequent Elements (LC #347)

```
Input: [1,1,1,2,2,3], k = 2  →  Output: [1, 2]
```

**Step 1:** Count frequencies. **Step 2:** Get top K by frequency.

Two clean approaches: heap or bucket sort.

In [ ]:
from collections import Counter
import heapq

# Approach 1: Heap — O(n log k)
def topKFrequent_heap(nums, k):
    count = Counter(nums)
    # nlargest by frequency
    return [num for num, freq in count.most_common(k)]

# Approach 2: Bucket Sort — O(n)
def topKFrequent_bucket(nums, k):
    count = Counter(nums)
    
    # Bucket: index = frequency, value = list of numbers with that freq
    buckets = [[] for _ in range(len(nums) + 1)]
    for num, freq in count.items():
        buckets[freq].append(num)
    
    # Collect from highest frequency bucket down
    result = []
    for freq in range(len(buckets) - 1, 0, -1):
        for num in buckets[freq]:
            result.append(num)
            if len(result) == k:
                return result
    return result

print(topKFrequent_heap([1,1,1,2,2,3], 2))    # [1, 2]
print(topKFrequent_bucket([1,1,1,2,2,3], 2))  # [1, 2]

---

## 🎯 Problem 3: K Closest Points to Origin (LC #973)

```
Input: points = [[1,3],[-2,2]], k = 1  →  Output: [[-2,2]]
(distance to origin: sqrt(1+9)=3.16 vs sqrt(4+4)=2.83)
```

Use a max-heap of size k (negate distances since Python has min-heap).

In [ ]:
import heapq

def kClosest(points, k):
    """
    Max-heap of size k (negate distances)
    Time: O(n log k), Space: O(k)
    """
    heap = []  # max-heap (negate distance)
    
    for x, y in points:
        dist = -(x*x + y*y)  # Negate for max-heap; no sqrt needed for comparison
        heapq.heappush(heap, (dist, x, y))
        if len(heap) > k:
            heapq.heappop(heap)  # Remove farthest point
    
    return [[x, y] for _, x, y in heap]

print(kClosest([[1,3],[-2,2]], 1))              # [[-2, 2]]
print(kClosest([[3,3],[5,-1],[-2,4]], 2))        # [[3,3],[-2,4]]

---

## 🗺️ Heap Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  HEAP PATTERNS                                                      ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. TOP K LARGEST: Min-heap of size K                               ║
║     Push all, pop when size > k → root is kth largest               ║
║                                                                      ║
║  2. TOP K SMALLEST: Max-heap of size K (negate values)              ║
║     Or just use min-heap and collect first K pops                    ║
║                                                                      ║
║  3. TOP K FREQUENT: Counter + heap or bucket sort                   ║
║     Bucket sort gives O(n) but uses O(n) space                      ║
║                                                                      ║
║  4. MERGE K SORTED: Push (val, list_idx, elem_idx) tuples          ║
║     Pop smallest, push next from same list                           ║
║                                                                      ║
║  5. MEDIAN STREAM: Two heaps (max-heap left, min-heap right)        ║
║     Balance sizes → median from tops                                 ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 215 | Kth Largest Element | Medium | Min-heap size k |
| 347 | Top K Frequent Elements | Medium | Counter + heap/bucket |
| 973 | K Closest Points to Origin | Medium | Max-heap size k |
| 23 | Merge K Sorted Lists | Hard | Min-heap |
| 295 | Find Median from Data Stream | Hard | Two heaps |
| 703 | Kth Largest Element in Stream | Easy | Min-heap size k |
| 692 | Top K Frequent Words | Medium | Counter + heap |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Python heapq = min-heap. Negate values for max-heap behavior.  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 "K largest" → min-heap of size K. Root = kth largest.          │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Heap push/pop = O(log k). Full pass = O(n log k).             │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Counter.most_common(k) is your friend for frequency problems.  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Bucket sort gives O(n) for top-K frequency problems.           │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Sliding Window

Fixed windows, variable windows, and the art of expanding and shrinking.

---

### 🎉 Heap Patterns Locked In!
See "Top K" or "Kth" in a problem? Heap. Done.